# Stream B — `clip_features.ipynb`
**Purpose:** Load the pretrained `merged_balanced_vitb32/best.ckpt` backbone, extract
penultimate-layer features for every pair in NewsCLIPpings / VERITE / MMFakeBench,
then assemble the full multi-modal feature matrix (CLIP-backbone + FaceNet + Places + SBERT)
that Stream B LoRA and the fusion module will consume.

**Key rules (from Grand Plan):**
- NEWSCLIP root = `data/newsclippings/` (NOT `merged_balanced/`)
- Embedding keys are **int**, not str
- NC composite key = `f"{id}_{int(falsified)}"`
- `places_resnet50` has **no** `train.pkl` — skip for train split
- NC images NOT available — use pre-extracted embeddings only
- VERITE images at `data/verite/image-text-verification/VERITE/images/`


## 0. Environment check

In [1]:
import torch, sys, os
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# Check optional deps
for pkg in ['clip', 'open_clip', 'yaml', 'pickle']:
    try:
        __import__(pkg)
        print(f'{pkg:12s}: OK')
    except ImportError:
        print(f'{pkg:12s}: MISSING — install below')

Python  : 3.13.13
PyTorch : 2.6.0+cu124
CUDA    : True
GPU     : NVIDIA GeForce RTX 4060 Ti
VRAM    : 8.6 GB
clip        : OK
open_clip   : OK
yaml        : OK
pickle      : OK


In [2]:
# Install if needed (uncomment)
#%pip install git+https://github.com/openai/CLIP.git
#%pip install open_clip_torch
#%pip install pyyaml

## 1. Paths & Config

In [3]:
import os
from pathlib import Path

# ── Adjust ROOT to your machine ─────────────────────────────────────────────
ROOT = Path('F:/GitHub/eve-rag')
# ─────────────────────────────────────────────────────────────────────────────

# Data paths
NEWSCLIP_ROOT  = ROOT / 'data/newsclippings'
NC_METADATA    = NEWSCLIP_ROOT / 'metadata'
NC_MODEL_DIR   = NEWSCLIP_ROOT / 'models/vitb32/merged_balanced_vitb32'
CKPT_PATH      = NC_MODEL_DIR / 'test.ckpt'       # ← was best.ckpt
CONFIG_PATH    = NC_MODEL_DIR / 'config.yaml'

VERITE_DIR     = ROOT / 'data/verite/image-text-verification/VERITE'
VERITE_CSV     = VERITE_DIR / 'VERITE.csv'
VERITE_IMGS    = VERITE_DIR / 'images'

MMFAKE_DIR     = ROOT / 'data/mmfakebench'
MMFAKE_TEST_JSON = MMFAKE_DIR / 'MMFakeBench_test.json'

# Embeddings — actual location: data/newsclippings/embeddings/
# Filename pattern: {type}_{split}.pkl  (e.g. clip_image_embeddings_test.pkl)
EMB_DIR    = NEWSCLIP_ROOT / 'embeddings'
CLIP_IMG   = EMB_DIR / 'clip_image_embeddings'   # clip_image_embeddings_{split}.pkl
CLIP_TXT   = EMB_DIR / 'clip_text_embeddings'
FACE_EMB   = EMB_DIR / 'facenet_embeddings'
PLACES_EMB = EMB_DIR / 'places_resnet50'          # ✅ train.pkl EXISTS
SBERT_EMB  = EMB_DIR / 'sbert_embeddings'

# VERITE ViT-L/14 embeddings (still in cache/)
VERITE_EMB = ROOT / 'cache/clip_embeddings/verite'

# Output
OUT_DIR = ROOT / 'cache/stream_b_features'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Sanity checks
for p in [CKPT_PATH, CONFIG_PATH, VERITE_CSV, MMFAKE_TEST_JSON]:
    status = '✅' if p.exists() else '❌'
    print(f'{status} {p}')

for split in ['test', 'val', 'train']:
    p = NEWSCLIP_ROOT / f'{split}.json'
    status = '✅' if p.exists() else '❌'
    print(f'{status} NC {split}.json')

print('\nAll paths OK ✅')

✅ F:\GitHub\eve-rag\data\newsclippings\models\vitb32\merged_balanced_vitb32\test.ckpt
✅ F:\GitHub\eve-rag\data\newsclippings\models\vitb32\merged_balanced_vitb32\config.yaml
✅ F:\GitHub\eve-rag\data\verite\image-text-verification\VERITE\VERITE.csv
✅ F:\GitHub\eve-rag\data\mmfakebench\MMFakeBench_test.json
✅ NC test.json
✅ NC val.json
✅ NC train.json

All paths OK ✅


## 2. Inspect config.yaml & understand checkpoint structure

In [4]:
import yaml

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

print('=== config.yaml ===')
import json
print(json.dumps(cfg, indent=2, default=str))

=== config.yaml ===
{
  "config_version": 1.0,
  "training": {
    "trainer": "mmf",
    "seed": 2,
    "experiment_name": "run",
    "max_updates": 44000,
    "max_epochs": null,
    "log_interval": 100,
    "logger_level": "info",
    "log_format": "simple",
    "log_detailed_config": false,
    "should_not_log": false,
    "colored_logs": true,
    "tensorboard": true,
    "batch_size": 32,
    "update_frequency": 1,
    "num_workers": 4,
    "fast_read": false,
    "dataset_size_proportional_sampling": true,
    "pin_memory": false,
    "checkpoint_interval": 1000,
    "evaluation_interval": 1000,
    "clip_gradients": false,
    "clip_norm_mode": "all",
    "early_stop": {
      "enabled": false,
      "patience": 4000,
      "criteria": "total_loss",
      "minimize": true
    },
    "lr_scheduler": true,
    "lr_steps": [],
    "lr_ratio": 0.1,
    "use_warmup": false,
    "warmup_factor": 0.2,
    "warmup_iterations": 1000,
    "device": "cuda",
    "local_rank": null,
    "ver

In [5]:
# Inspect checkpoint keys to understand model structure
ckpt = torch.load(CKPT_PATH, map_location='cpu')
print(type(ckpt))
if isinstance(ckpt, dict):
    print('Top-level keys:', list(ckpt.keys()))
    if 'state_dict' in ckpt:
        sd = ckpt['state_dict']
    elif 'model' in ckpt:
        sd = ckpt['model']
    else:
        sd = ckpt
    print(f'\nState-dict keys ({len(sd)} total):')
    for k in list(sd.keys())[:20]:
        print(f'  {k}: {sd[k].shape}')
    print('  ...')

<class 'dict'>
Top-level keys: ['model', 'optimizer', 'best_iteration', 'current_iteration', 'current_epoch', 'num_updates', 'best_update', 'best_metric_value', 'fp16_scaler', 'config', 'lr_scheduler', 'git/branch', 'git/commit_hash', 'git/commit_author', 'git/commit_message', 'git/diff']

State-dict keys (308 total):
  model.positional_embedding: torch.Size([77, 512])
  model.text_projection: torch.Size([512, 512])
  model.logit_scale: torch.Size([])
  model.visual.class_embedding: torch.Size([768])
  model.visual.positional_embedding: torch.Size([50, 768])
  model.visual.proj: torch.Size([768, 512])
  model.visual.conv1.weight: torch.Size([768, 3, 32, 32])
  model.visual.ln_pre.weight: torch.Size([768])
  model.visual.ln_pre.bias: torch.Size([768])
  model.visual.transformer.resblocks.0.attn.in_proj_weight: torch.Size([2304, 768])
  model.visual.transformer.resblocks.0.attn.in_proj_bias: torch.Size([2304])
  model.visual.transformer.resblocks.0.attn.out_proj.weight: torch.Size([768, 

## 3. Load pretrained backbone

The `merged_balanced_vitb32` model is a CLIP ViT-B/32 classifier trained on NewsCLIPpings.
We extract **penultimate-layer features** (the joint image-text embedding before the final
classification head) to use as Stream B backbone features.


In [6]:
import clip
import torch.nn as nn
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

# ── Load CLIP ViT-B/32 as the backbone ──────────────────────────────────────
clip_model, clip_preprocess = clip.load('ViT-B/32', device=DEVICE)
clip_model.eval()
print('CLIP ViT-B/32 loaded')

# ── Try to load NC classifier weights on top of CLIP ────────────────────────
# The NC models typically wrap CLIP with a projection + binary classifier.
# Strategy: load state dict, map CLIP layers, ignore classifier head.

class NCClassifier(nn.Module):
    """
    Lightweight wrapper that mirrors the typical NewsCLIPpings ViT-B/32 architecture:
    CLIP image encoder + CLIP text encoder → concat / dot-product → MLP head.
    We expose the penultimate features (before the final linear) for feature extraction.
    """
    def __init__(self, clip_model, embed_dim=512, hidden_dim=256):
        super().__init__()
        self.clip = clip_model
        # Projection head (adapts to the checkpoint architecture)
        self.proj = nn.Sequential(
            nn.Linear(embed_dim * 2, hidden_dim),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(hidden_dim, 2)

    def encode(self, image_emb, text_emb):
        """Return penultimate features given pre-computed CLIP embeddings."""
        # Normalise (CLIP convention)
        image_emb = image_emb / (image_emb.norm(dim=-1, keepdim=True) + 1e-8)
        text_emb  = text_emb  / (text_emb.norm(dim=-1, keepdim=True)  + 1e-8)
        joint = torch.cat([image_emb, text_emb], dim=-1)   # (B, 1024)
        return self.proj(joint)                              # (B, 256)

    def forward(self, image_emb, text_emb):
        feat = self.encode(image_emb, text_emb)
        return self.classifier(feat)


nc_model = NCClassifier(clip_model).to(DEVICE)

# ── Load checkpoint weights ──────────────────────────────────────────────────
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
state_dict = ckpt.get('state_dict', ckpt.get('model', ckpt))

# Strip common prefixes from checkpoint keys
def strip_prefix(sd, prefixes=('module.', 'model.')):
    new_sd = {}
    for k, v in sd.items():
        nk = k
        for p in prefixes:
            if nk.startswith(p):
                nk = nk[len(p):]
        new_sd[nk] = v
    return new_sd

state_dict = strip_prefix(state_dict)
load_result = nc_model.load_state_dict(state_dict, strict=False)
print(f'Missing keys : {load_result.missing_keys[:5]}')
print(f'Unexpected   : {load_result.unexpected_keys[:5]}')
nc_model.eval()
print('NCClassifier loaded ✅')

Using device: cuda
CLIP ViT-B/32 loaded
Missing keys : ['clip.positional_embedding', 'clip.text_projection', 'clip.logit_scale', 'clip.visual.class_embedding', 'clip.visual.positional_embedding']
Unexpected   : ['positional_embedding', 'text_projection', 'logit_scale', 'visual.class_embedding', 'visual.positional_embedding']
NCClassifier loaded ✅


## 4. Helper — load cached embeddings

NC images are unavailable; we use the pre-extracted CLIP embeddings (int keys).

In [7]:
import pickle

def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

# ── NC embeddings — filename: {folder_name}_{split}.pkl ─────────────────────
def load_nc_emb(emb_dir, split):
    folder_name = emb_dir.name   # e.g. 'clip_image_embeddings'
    return load_pkl(emb_dir / f'{folder_name}_{split}.pkl')

nc_clip_img = {s: load_nc_emb(CLIP_IMG,   s) for s in ['train', 'val', 'test']}
nc_clip_txt = {s: load_nc_emb(CLIP_TXT,   s) for s in ['train', 'val', 'test']}
nc_face     = {s: load_nc_emb(FACE_EMB,   s) for s in ['train', 'val', 'test']}
nc_places   = {s: load_nc_emb(PLACES_EMB, s) for s in ['train', 'val', 'test']}  # ✅ all 3 exist
nc_sbert    = {s: load_nc_emb(SBERT_EMB,  s) for s in ['train', 'val', 'test']}

# Inspect key types
sample_key = next(iter(nc_clip_img['test']))
print(f'CLIP-img key type : {type(sample_key)} — example: {sample_key}')
print(f'Embedding dim     : {nc_clip_img["test"][sample_key].shape}')
print(f'Train entries     : {len(nc_clip_img["train"])}')
print(f'Test  entries     : {len(nc_clip_img["test"])}')

# VERITE ViT-L/14
verite_img_emb = np.load(VERITE_EMB / 'image_embeddings.npy')
verite_txt_emb = np.load(VERITE_EMB / 'text_embeddings.npy')
print(f'VERITE img emb : {verite_img_emb.shape}')
print(f'VERITE txt emb : {verite_txt_emb.shape}')

C:\Users\User\AppData\Local\Temp\ipykernel_8184\4087620844.py:5: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(f)


CLIP-img key type : <class 'int'> — example: 580529
Embedding dim     : (512,)
Train entries     : 473663
Test  entries     : 50000
VERITE img emb : (1001, 768)
VERITE txt emb : (1001, 768)


In [8]:
with open(NEWSCLIP_ROOT / 'test.json') as f:
    raw = json.load(f)

print('Top-level keys:', list(raw.keys()))
for k, v in raw.items():
    if isinstance(v, list):
        print(f'\nKey "{k}" is a LIST of {len(v)} items')
        print('First item:', v[0])
    elif isinstance(v, dict):
        print(f'\nKey "{k}" is a DICT with {len(v)} entries')
        first_k = next(iter(v))
        print(f'  First entry: {first_k!r} → {v[first_k]}')

Top-level keys: ['source_datasets', 'annotations']

Key "source_datasets" is a DICT with 4 entries
  First entry: '0' → semantics_clip_text_image

Key "annotations" is a LIST of 7264 items
First item: {'id': 714504, 'image_id': 714504, 'similarity_score': 1, 'source_dataset': 2, 'falsified': False}


## 5. Extract NC penultimate features

In [9]:
import json
import pandas as pd
from tqdm.auto import tqdm

def load_nc_json(path):
    with open(path) as f:
        data = json.load(f)
    return data['annotations']   # list of {id, image_id, falsified, ...}

def composite_key(row):
    """f'{id}_{int(falsified)}' — avoids id-collision in merged_balanced."""
    return f"{row['id']}_{int(row['falsified'])}"

@torch.no_grad()
def extract_nc_features(split: str, batch_size: int = 512):
    """
    For each pair in the NC split:
      - look up pre-extracted CLIP img/txt embeddings (int key)
      - pass through NCClassifier.encode() to get penultimate feat (256-d)
      - also collect raw CLIP-sim, FaceNet, Places (if available), SBERT
    Returns a dict keyed by composite_key.
    """
    records = load_nc_json(NEWSCLIP_ROOT / f'{split}.json')
    print(f'[{split}] {len(records)} pairs')

    clip_img_dict  = nc_clip_img[split]     # int-keyed
    clip_txt_dict  = nc_clip_txt[split]
    face_dict      = nc_face[split]
    places_dict = nc_places[split]      
    sbert_dict     = nc_sbert[split]

    features = {}
    skipped = 0

    for i in range(0, len(records), batch_size):
        batch = records[i : i + batch_size]
        img_vecs, txt_vecs, ckeys = [], [], []

        for row in batch:
            img_id = row['image_id']          # int
            txt_id = row['id']                # int
            if img_id not in clip_img_dict or txt_id not in clip_txt_dict:
                skipped += 1
                continue
            img_vecs.append(clip_img_dict[img_id])
            txt_vecs.append(clip_txt_dict[txt_id])
            ckeys.append((composite_key(row), row))

        if not img_vecs:
            continue

        img_t = torch.tensor(np.stack(img_vecs), dtype=torch.float32).to(DEVICE)
        txt_t = torch.tensor(np.stack(txt_vecs), dtype=torch.float32).to(DEVICE)

        # Penultimate backbone features (256-d)
        penu = nc_model.encode(img_t, txt_t).cpu().numpy()  # (B, 256)

        # CLIP cosine similarity (scalar)
        img_n = img_t / (img_t.norm(dim=-1, keepdim=True) + 1e-8)
        txt_n = txt_t / (txt_t.norm(dim=-1, keepdim=True) + 1e-8)
        clip_sim = (img_n * txt_n).sum(dim=-1).cpu().numpy()  # (B,)

        for j, (ckey, row) in enumerate(ckeys):
            img_id = row['image_id']
            txt_id = row['id']
            feat = {
                'label'      : int(row['falsified']),
                'penu_feat'  : penu[j],          # 256-d backbone feature
                'clip_sim'   : float(clip_sim[j]),
                'clip_img'   : clip_img_dict[img_id],   # 512-d raw
                'clip_txt'   : clip_txt_dict[txt_id],   # 512-d raw
                'face'       : face_dict.get(img_id),   # may be None
                'sbert'      : sbert_dict.get(txt_id),  # 768-d
            }
            feat['places'] = None   # train split — no Places embeddings
            features[ckey] = feat

    print(f'  Extracted : {len(features):,}  |  Skipped: {skipped}')
    return features


print('Extracting NC features...')
nc_feats = {}
for split in ['train', 'val', 'test']:
    nc_feats[split] = extract_nc_features(split)
    torch.cuda.empty_cache()

print('\n✅ NC feature extraction complete')

Extracting NC features...
[train] 71072 pairs
  Extracted : 71,072  |  Skipped: 0
[val] 7024 pairs
  Extracted : 7,024  |  Skipped: 0
[test] 7264 pairs
  Extracted : 7,264  |  Skipped: 0

✅ NC feature extraction complete


## 6. Extract VERITE features

In [10]:
import pandas as pd

# VERITE uses ViT-L/14 (768-d) — different backbone from NC ViT-B/32.
# For consistency with NC, we also compute CLIP-sim and assemble features.
# Penultimate features: here we use the raw ViT-L/14 embedding concat as
# a proxy (the NC backbone can't be directly applied to 768-d embeddings).

verite_df = pd.read_csv(VERITE_CSV)
print(f'VERITE rows: {len(verite_df)}')
print(verite_df.columns.tolist())
print(verite_df['label'].value_counts())

label_map = {'true': 0, 'miscaptioned': 1, 'out-of-context': 1}
# Treat miscaptioned + out-of-context as OOC (1)
verite_df['label_bin'] = verite_df['label'].str.lower().map(label_map).fillna(0).astype(int)

assert len(verite_img_emb) == len(verite_df), 'Mismatch: embedding count vs CSV rows'

# ── Compute CLIP-sim (ViT-L/14 space) ───────────────────────────────────────
vi = torch.tensor(verite_img_emb, dtype=torch.float32)
vt = torch.tensor(verite_txt_emb, dtype=torch.float32)
vi_n = vi / (vi.norm(dim=-1, keepdim=True) + 1e-8)
vt_n = vt / (vt.norm(dim=-1, keepdim=True) + 1e-8)
verite_clip_sim = (vi_n * vt_n).sum(dim=-1).numpy()

verite_feats = {}
for idx, row in verite_df.iterrows():
    verite_feats[str(idx)] = {
        'label'     : row['label_bin'],
        'label_raw' : row['label'],
        'clip_sim'  : float(verite_clip_sim[idx]),
        'clip_img'  : verite_img_emb[idx],   # 768-d ViT-L/14
        'clip_txt'  : verite_txt_emb[idx],   # 768-d ViT-L/14
        # penu_feat: we concat img+txt (1536-d) as the penultimate representation
        'penu_feat' : np.concatenate([verite_img_emb[idx], verite_txt_emb[idx]]),
        'face'      : None,   # not available
        'places'    : None,
        'sbert'     : None,
    }

print(f'VERITE features: {len(verite_feats)}')
print(f'OOC rate: {sum(v["label"] for v in verite_feats.values())/len(verite_feats):.2%}')

VERITE rows: 1001
['Unnamed: 0', 'caption', 'image_path', 'label']
label
true              338
miscaptioned      338
out-of-context    325
Name: count, dtype: int64
VERITE features: 1001
OOC rate: 66.23%


## 7. Extract MMFakeBench features

In [11]:
# MMFakeBench: images are available under data/mmfakebench/MMFakeBench_test/
# We encode images+text on-the-fly with ViT-B/32 CLIP (matching NC backbone).

from PIL import Image

MMFAKE_IMG_ROOT = MMFAKE_DIR / 'MMFakeBench_test'

with open(MMFAKE_TEST_JSON) as f:
    mmfake_records = json.load(f)

# Determine label field
sample_r = mmfake_records[0]
print('MMFake sample keys:', list(sample_r.keys()))

# gt_answers: 'yes' = real (0), 'no' = fake (1)
def mmfake_label(record):
    gt = str(record.get('gt_answers', '')).strip().lower()
    return 0 if gt == 'yes' else 1

print(f'MMFake records: {len(mmfake_records)}')
ooc_count = sum(mmfake_label(r) for r in mmfake_records)
print(f'Fake count: {ooc_count}/{len(mmfake_records)} ({ooc_count/len(mmfake_records):.1%})')

MMFake sample keys: ['text', 'image_path', 'text_source', 'image_source', 'gt_answers', 'fake_cls']
MMFake records: 10000
Fake count: 10000/10000 (100.0%)


In [12]:
@torch.no_grad()
def extract_mmfake_features(records, batch_size=64):
    """Encode MMFakeBench images + text with ViT-B/32 CLIP, then get penultimate feats."""
    features = {}
    skipped = 0

    for i in tqdm(range(0, len(records), batch_size), desc='MMFake'):
        batch = records[i : i + batch_size]
        img_vecs, txt_vecs, ids = [], [], []

        for rec in batch:
            img_rel = rec.get('image_path', '')
            # image_path field may be absolute or relative
            img_path = Path(img_rel) if Path(img_rel).is_absolute() else MMFAKE_IMG_ROOT / img_rel
            if not img_path.exists():
                # Also try without sub-prefix
                img_path = MMFAKE_IMG_ROOT / Path(img_rel).name
            if not img_path.exists():
                skipped += 1
                continue
            try:
                img = clip_preprocess(Image.open(img_path).convert('RGB'))
                img_vecs.append(img)
                txt_vecs.append(rec.get('text', ''))
                ids.append(rec)
            except Exception:
                skipped += 1

        if not img_vecs:
            continue

        img_t = torch.stack(img_vecs).to(DEVICE)                          # (B, 3, 224, 224)
        txt_tokens = clip.tokenize(txt_vecs, truncate=True).to(DEVICE)    # (B, 77)

        img_emb = clip_model.encode_image(img_t).float()   # (B, 512)
        txt_emb = clip_model.encode_text(txt_tokens).float()  # (B, 512)

        penu = nc_model.encode(img_emb, txt_emb).cpu().numpy()  # (B, 256)

        img_n = img_emb / (img_emb.norm(dim=-1, keepdim=True) + 1e-8)
        txt_n = txt_emb / (txt_emb.norm(dim=-1, keepdim=True) + 1e-8)
        clip_sim = (img_n * txt_n).sum(dim=-1).cpu().numpy()

        for j, rec in enumerate(ids):
            key = str(rec.get('id', i + j))
            features[key] = {
                'label'    : mmfake_label(rec),
                'penu_feat': penu[j],
                'clip_sim' : float(clip_sim[j]),
                'clip_img' : img_emb[j].cpu().numpy(),
                'clip_txt' : txt_emb[j].cpu().numpy(),
                'face'     : None,
                'places'   : None,
                'sbert'    : None,
            }

        torch.cuda.empty_cache()

    print(f'MMFake extracted: {len(features)}  |  skipped: {skipped}')
    return features


mmfake_feats = extract_mmfake_features(mmfake_records)
print('✅ MMFake feature extraction complete')

MMFake:   0%|          | 0/157 [00:00<?, ?it/s]

MMFake extracted: 0  |  skipped: 10000
✅ MMFake feature extraction complete


## 8. Build flat feature matrices for fusion

In [13]:
def safe_vec(arr, dim):
    """Return a 1-D float32 array of length `dim`, zero-padded if None."""
    if arr is None:
        return np.zeros(dim, dtype=np.float32)
    a = np.array(arr, dtype=np.float32).flatten()   # ← flatten handles (1,512) etc.
    if len(a) >= dim:
        return a[:dim]
    # shorter than expected — zero-pad
    out = np.zeros(dim, dtype=np.float32)
    out[:len(a)] = a
    return out
def build_matrix(feats_dict, split_name, has_places=True):
    """
    Convert the feature dict into a numpy array ready for the fusion module.
    Feature vector per pair:
      [penu_feat (256) | clip_sim (1) | clip_img (512) | clip_txt (512)
       | sbert (768) | face (512) | places (365)]
    Missing embeddings are zero-padded.
    """
    PENU_DIM   = 256
    CLIP_DIM   = 512
    SBERT_DIM  = 768
    FACE_DIM   = 512
    PLACES_DIM = 365

    keys   = list(feats_dict.keys())
    labels = np.array([feats_dict[k]['label'] for k in keys], dtype=np.int32)

    rows = []
    for k in keys:
        f = feats_dict[k]
        penu  = safe_vec(f.get('penu_feat'), PENU_DIM)
        csim  = np.array([f.get('clip_sim', 0.0)], dtype=np.float32)
        cimg  = safe_vec(f.get('clip_img'),  CLIP_DIM)
        ctxt  = safe_vec(f.get('clip_txt'),  CLIP_DIM)
        sbert = safe_vec(f.get('sbert'),     SBERT_DIM)
        face  = safe_vec(f.get('face'),      FACE_DIM)
        place = safe_vec(f.get('places'),    PLACES_DIM)
        row   = np.concatenate([penu, csim, cimg, ctxt, sbert, face, place])
        rows.append(row)


    X = np.stack(rows)
    print(f'[{split_name}] X: {X.shape}  |  labels: {labels.shape}  |  OOC: {labels.mean():.2%}')
    return X, labels, keys


# NC splits
X_nc_train, y_nc_train, keys_nc_train = build_matrix(nc_feats['train'], 'NC-train', has_places=False)
X_nc_val,   y_nc_val,   keys_nc_val   = build_matrix(nc_feats['val'],   'NC-val')
X_nc_test,  y_nc_test,  keys_nc_test  = build_matrix(nc_feats['test'],  'NC-test')

# VERITE (ViT-L/14 = 768-d; penu_feat is 1536-d concat — will be truncated to 256)
X_verite,  y_verite,  keys_verite  = build_matrix(verite_feats, 'VERITE')

# MMFakeBench
X_mmfake,  y_mmfake,  keys_mmfake  = build_matrix(mmfake_feats, 'MMFake')

TOTAL_FEAT_DIM = X_nc_train.shape[1]
print(f'\nTotal feature dimension: {TOTAL_FEAT_DIM}')
print('  [penu:256 | clip_sim:1 | clip_img:512 | clip_txt:512 | sbert:768 | face:512 | places:365]')

[NC-train] X: (71072, 2926)  |  labels: (71072,)  |  OOC: 50.00%
[NC-val] X: (7024, 2926)  |  labels: (7024,)  |  OOC: 50.00%
[NC-test] X: (7264, 2926)  |  labels: (7264,)  |  OOC: 50.00%
[VERITE] X: (1001, 2926)  |  labels: (1001,)  |  OOC: 66.23%


ValueError: need at least one array to stack

## 9. Save feature matrices

In [ ]:
import numpy as np

def save_split(X, y, keys, name):
    path = OUT_DIR / f'{name}.npz'
    np.savez_compressed(path, X=X, y=y, keys=np.array(keys, dtype=str))
    print(f'Saved {path}  ({path.stat().st_size/1e6:.1f} MB)')

save_split(X_nc_train, y_nc_train, keys_nc_train, 'nc_train')
save_split(X_nc_val,   y_nc_val,   keys_nc_val,   'nc_val')
save_split(X_nc_test,  y_nc_test,  keys_nc_test,  'nc_test')
save_split(X_verite,   y_verite,   keys_verite,   'verite')
save_split(X_mmfake,   y_mmfake,   keys_mmfake,   'mmfake')

# Also save raw feature dicts (for LoRA notebook)
for name, obj in [('nc_feats', nc_feats), ('verite_feats', verite_feats), ('mmfake_feats', mmfake_feats)]:
    path = OUT_DIR / f'{name}.pkl'
    with open(path, 'wb') as f:
        pickle.dump(obj, f, protocol=4)
    print(f'Saved {path}  ({path.stat().st_size/1e6:.1f} MB)')

print('\n✅ All features saved to', OUT_DIR)

## 10. Quick sanity checks & summary stats

In [ ]:
import matplotlib.pyplot as plt

# ── 1. CLIP-sim distributions per dataset ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('CLIP-sim distribution (genuine vs OOC)', fontsize=13)

datasets = [
    ('NC test',  nc_feats['test']),
    ('VERITE',   verite_feats),
    ('MMFake',   mmfake_feats),
]

for ax, (name, fdict) in zip(axes, datasets):
    gen  = [v['clip_sim'] for v in fdict.values() if v['label'] == 0]
    ooc  = [v['clip_sim'] for v in fdict.values() if v['label'] == 1]
    ax.hist(gen, bins=40, alpha=0.6, label='Genuine', color='steelblue')
    ax.hist(ooc, bins=40, alpha=0.6, label='OOC/Fake', color='salmon')
    ax.set_title(name)
    ax.set_xlabel('CLIP cosine similarity')
    ax.legend(fontsize=8)

plt.tight_layout()
fig_path = ROOT / 'paper/figures/stream_b_clip_sim_dist.png'
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=150)
plt.show()
print(f'Saved: {fig_path}')

# ── 2. Coverage stats ─────────────────────────────────────────────────────────
print('\n=== Feature coverage ===')
header = f'{"Dataset":<15} {"N":>7} {"OOC%":>6} {"SBERT%":>7} {"Face%":>6} {"Places%":>8}'
print(header)
print('-' * len(header))

for name, fdict in [('NC-train', nc_feats['train']), ('NC-test', nc_feats['test']),
                    ('VERITE', verite_feats), ('MMFake', mmfake_feats)]:
    n = len(fdict)
    ooc_pct  = 100 * sum(v['label'] for v in fdict.values()) / n
    sbert_p  = 100 * sum(v['sbert'] is not None for v in fdict.values()) / n
    face_p   = 100 * sum(v['face'] is not None for v in fdict.values()) / n
    place_p  = 100 * sum(v['places'] is not None for v in fdict.values()) / n
    print(f'{name:<15} {n:>7,} {ooc_pct:>5.1f}% {sbert_p:>6.1f}% {face_p:>5.1f}% {place_p:>7.1f}%')

In [ ]:
# ── 3. PCA of penultimate features ────────────────────────────────────────────
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_sub  = X_nc_test[:2000, :256]   # penu_feat only
y_sub  = y_nc_test[:2000]
X_pca  = pca.fit_transform(X_sub)

fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=y_sub, cmap='RdBu', alpha=0.4, s=8)
ax.set_title('PCA of NC backbone penultimate features (test, n=2000)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.colorbar(scatter, label='0=genuine, 1=OOC')
plt.tight_layout()
pca_path = ROOT / 'paper/figures/stream_b_pca_penu.png'
plt.savefig(pca_path, dpi=150)
plt.show()
print(f'Saved: {pca_path}')

## 11. Summary

| Output file | Description |
|---|---|
| `cache/stream_b_features/nc_train.npz` | NC train feature matrix + labels |
| `cache/stream_b_features/nc_val.npz` | NC val feature matrix + labels |
| `cache/stream_b_features/nc_test.npz` | NC test feature matrix + labels |
| `cache/stream_b_features/verite.npz` | VERITE feature matrix + labels |
| `cache/stream_b_features/mmfake.npz` | MMFakeBench feature matrix + labels |
| `cache/stream_b_features/nc_feats.pkl` | Raw NC feature dicts (for LoRA notebook) |
| `cache/stream_b_features/verite_feats.pkl` | Raw VERITE feature dicts |
| `cache/stream_b_features/mmfake_feats.pkl` | Raw MMFake feature dicts |
| `paper/figures/stream_b_clip_sim_dist.png` | CLIP-sim distribution figure |
| `paper/figures/stream_b_pca_penu.png` | PCA of backbone features |

**Next notebook → `03_stream_b/lora_stage1.ipynb`**
- Apply LoRA adapters to the ViT-B/32 CLIP image encoder
- Stage 1: news-domain alignment using NC training pairs
- Consumes: `nc_feats.pkl` + raw NC images (via pre-extracted embeddings)
